# 02: RUL予測ベースラインの学習

FD001を使い、Dummy、Ridge、Random Forestを比較して公式testを評価します。

各セルを`Shift+Enter`で順番に実行してください。ハイパーパラメータを変えた実験は、設定セルから再実行できます。

## 1. プロジェクトと実験条件の設定

In [ ]:
from pathlib import Path
import sys

# VS Codeの設定により通常はリポジトリ直下が作業フォルダになります。
# 念のため、notebooksフォルダが作業場所になっていても親を探せるようにします。
PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "CMAPSSData").exists():
    PROJECT_DIR = PROJECT_DIR.parent

if not (PROJECT_DIR / "CMAPSSData").exists():
    raise FileNotFoundError("CMAPSSDataが見つかりません。VS CodeでC-MAPSSフォルダを開いてください。")

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATASET = "FD001"  # FD002、FD003、FD004も指定できます。
RUL_CAP = 125

print(f"project: {PROJECT_DIR}")
print(f"dataset: {DATASET}")


# ここを変更するとモデルの条件を対話的に試せます。
RANDOM_STATE = 42
ROLLING_WINDOW = 5
N_TREES = 250
VALIDATION_FRACTIONS = (0.5, 0.7, 0.9)

## 2. ライブラリと共通関数の読み込み

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from cmapss.data import add_rul, load_split
from cmapss.features import add_rolling_features, feature_columns, select_active_sensors
from cmapss.metrics import regression_metrics

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", context="notebook")

## 3. データ読み込みと特徴量作成

In [ ]:
train, test, test_rul = load_split(PROJECT_DIR / "CMAPSSData", DATASET)
train = add_rul(train, cap=RUL_CAP)
active_sensors = select_active_sensors(train)

train_features = add_rolling_features(train, active_sensors, ROLLING_WINDOW)
test_features = add_rolling_features(test, active_sensors, ROLLING_WINDOW)
features = feature_columns(train_features, active_sensors, ROLLING_WINDOW)

print(f"train rows: {len(train_features):,}")
print(f"features  : {len(features)}")
features[:10]

## 4. エンジン単位で学習・検証へ分割

同じエンジンの行を学習と検証の両方へ入れるとデータリークになります。そのため100台を80台と20台に分けます。

In [ ]:
random = np.random.default_rng(RANDOM_STATE)
units = train_features["unit_id"].unique()
random.shuffle(units)
split_at = int(len(units) * 0.8)
fit_units = units[:split_at]
validation_units = units[split_at:]

fit_rows = train_features[train_features["unit_id"].isin(fit_units)].copy()
validation_rows = train_features[train_features["unit_id"].isin(validation_units)].copy()

print(f"学習エンジン: {len(fit_units)}台")
print(f"検証エンジン: {len(validation_units)}台")

## 5. 擬似的なtest終端を作成

実際のtestは故障前で観測が終了します。その状況を再現するため、検証エンジンの寿命50%、70%、90%時点だけを評価対象にします。

In [ ]:
endpoint_parts = []
for fraction in VALIDATION_FRACTIONS:
    lifetimes = validation_rows.groupby("unit_id")["cycle"].max()
    cutoffs = np.floor(lifetimes * fraction).astype(int).clip(lower=1)
    keys = pd.MultiIndex.from_arrays([cutoffs.index, cutoffs.values])
    mask = validation_rows.set_index(["unit_id", "cycle"]).index.isin(keys)
    part = validation_rows.loc[mask].copy()
    part["cutoff_fraction"] = fraction
    endpoint_parts.append(part)

validation_endpoints = pd.concat(endpoint_parts, ignore_index=True)
validation_endpoints[["unit_id", "cycle", "cutoff_fraction", "RUL_raw"]].head(10)

## 6. 比較するモデルを定義

In [ ]:
models = {
    # 全件に中央値を返すだけの、最低限超えるべき基準です。
    "Dummy median": DummyRegressor(strategy="median"),
    # 入力とRULの直線的な関係を学びます。
    "Ridge": Pipeline([
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ]),
    # 多数の決定木を組み合わせ、非線形な関係も学びます。
    "Random Forest": RandomForestRegressor(
        n_estimators=N_TREES,
        min_samples_leaf=3,
        max_features=0.7,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}
models

## 7. 学習と検証結果の比較

In [ ]:
validation_results = []
for name, model in models.items():
    model.fit(fit_rows[features], fit_rows["RUL"])
    predicted = np.clip(model.predict(validation_endpoints[features]), 0, RUL_CAP)
    validation_results.append({
        "model": name,
        **regression_metrics(validation_endpoints["RUL_raw"], predicted),
    })

validation_metrics = (
    pd.DataFrame(validation_results)
    .set_index("model")
    .sort_values("RMSE")
)
validation_metrics.round(3)

In [ ]:
best_name = validation_metrics.index[0]
best_model = models[best_name]
print(f"選択されたモデル: {best_name}")

if best_name == "Random Forest":
    importance = pd.Series(best_model.feature_importances_, index=features).nlargest(15)
    plt.figure(figsize=(9, 6))
    sns.barplot(x=importance.values, y=importance.index, color="#16a34a")
    plt.title("Random Forest feature importance")
    plt.xlabel("importance")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

## 8. 全trainで再学習し、公式testを評価

In [ ]:
best_model.fit(train_features[features], train_features["RUL"])

last_indices = test_features.groupby("unit_id")["cycle"].idxmax()
test_endpoints = test_features.loc[last_indices].sort_values("unit_id")
assert test_endpoints["unit_id"].tolist() == test_rul.index.tolist()

test_prediction = np.clip(best_model.predict(test_endpoints[features]), 0, RUL_CAP)
test_truth = test_rul["RUL"].to_numpy()
test_metrics = pd.DataFrame([{
    "dataset": DATASET,
    "model": best_name,
    **regression_metrics(test_truth, test_prediction),
}])
test_metrics.round(3)

In [ ]:
predictions = pd.DataFrame({
    "unit_id": test_endpoints["unit_id"].to_numpy(),
    "last_observed_cycle": test_endpoints["cycle"].to_numpy(),
    "RUL_true": test_truth,
    "RUL_pred": test_prediction,
    "error": test_prediction - test_truth,
})

figure, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(test_truth, test_prediction, alpha=0.7, color="#2563eb")
limit = max(test_truth.max(), test_prediction.max()) + 5
axes[0].plot([0, limit], [0, limit], "--", color="black", linewidth=1)
axes[0].set(xlabel="true RUL", ylabel="predicted RUL", title="Official test predictions")

sns.histplot(predictions["error"], bins=18, kde=True, ax=axes[1], color="#f97316")
axes[1].axvline(0, linestyle="--", color="black", linewidth=1)
axes[1].set(xlabel="prediction - truth", title="Prediction error")
plt.tight_layout()
plt.show()

## 9. 実験結果を保存

ハイパーパラメータを変更した場合は、保存先の`EXPERIMENT_NAME`も変更すると結果を比較しやすくなります。

In [ ]:
EXPERIMENT_NAME = "rf_default"
OUTPUT_DIR = PROJECT_DIR / "artifacts" / "notebook" / DATASET / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

validation_metrics.reset_index().to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
test_metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
joblib.dump({
    "model": best_model,
    "feature_columns": features,
    "active_sensors": active_sensors,
    "rul_cap": RUL_CAP,
}, OUTPUT_DIR / "baseline_model.joblib", compress=3)

print(f"保存先: {OUTPUT_DIR}")
sorted(path.name for path in OUTPUT_DIR.iterdir())

## 実験アイデア

- `N_TREES`を100、500へ変更する
- `ROLLING_WINDOW`を3、10、20へ変更する
- Ridgeの`alpha`を1、100へ変更する
- `DATASET`をFD002〜FD004へ変更する

設定を変えたら、特徴量作成またはモデル定義のセル以降を再実行してください。